In [27]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

OPENALEX_CSV = Path("openalex_works.csv")                 # <- set your filename
S2_CSV       = Path("semantic_scholar_results.csv")         # <- your existing file
OUT_CSV      = Path("stageA_combined_oa_s2.csv")


In [28]:
df_oa = pd.read_csv(OPENALEX_CSV)
df_s2 = pd.read_csv(S2_CSV)

print("OpenAlex:", df_oa.shape, "| cols:", list(df_oa.columns))
print("S2      :", df_s2.shape, "| cols:", list(df_s2.columns))

df_oa.head(1)

OpenAlex: (1625, 10) | cols: ['query', 'title', 'year', 'type', 'venue', 'cited_by', 'authors(first6)', 'doi', 'openalex_id', 'abstract']
S2      : (500, 10) | cols: ['query', 'paperId', 'title', 'year', 'venue', 'citationCount', 'authors(first6)', 'doi', 's2_url', 'abstract']


,query,title,year,type,venue,cited_by,authors(first6),doi,openalex_id,abstract
0,dynamic pricing definition concepts brick-and-...,Digital Transformation: An Overview of the Cur...,2021,article,SAGE Open,1044,Sascha Kraus; Paul Jones; Norbert Kailer; Alex...,https://doi.org/10.1177/21582440211047576,https://openalex.org/W3202582446,The increasing digitalization of economies has...


In [29]:
df_s2.head(1)

,query,paperId,title,year,venue,citationCount,authors(first6),doi,s2_url,abstract
0,dynamic pricing theory conceptual framework re...,10b032f405418862ee6cbabac505542398709ffa,Dynamic Retail Pricing via Q-Learning - A Rein...,2025.0,2025 1st International Conference on AIML-Appl...,12,Mohit Apte; rd Pranav Datar; Ketan Kale; Dr P....,10.1109/ICAET63349.2025.10932302,https://www.semanticscholar.org/paper/10b032f4...,This study examines how a reinforcement learni...


In [30]:
def standardize_openalex(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["source"] = "openalex"
    d["source_id"] = d.get("openalex_id")
    d["citation_count"] = d.get("cited_by")
    # keep your columns; make sure "abstract" exists
    for col in ["abstract", "doi", "title", "year", "venue", "type", "authors(first6)", "query"]:
        if col not in d.columns:
            d[col] = np.nan
    return d[[
        "source","source_id","query","title","year","venue","type",
        "authors(first6)","doi","citation_count","abstract","openalex_id"
    ]]

def standardize_s2(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["source"] = "semantic_scholar"
    d["source_id"] = d.get("paperId")
    d["citation_count"] = d.get("citationCount")
    for col in ["abstract", "doi", "title", "year", "venue", "authors(first6)", "query", "paperId", "s2_url"]:
        if col not in d.columns:
            d[col] = np.nan
    d["type"] = np.nan  # S2 search doesn't consistently return type
    d["openalex_id"] = np.nan
    return d[[
        "source","source_id","query","title","year","venue","type",
        "authors(first6)","doi","citation_count","abstract","paperId","s2_url"
    ]]

oa_std = standardize_openalex(df_oa)
s2_std = standardize_s2(df_s2)

combined_raw = pd.concat([oa_std, s2_std], ignore_index=True)
print("Combined raw:", combined_raw.shape)
combined_raw.head(5)


Combined raw: (2125, 14)


,source,source_id,query,title,year,venue,type,authors(first6),doi,citation_count,abstract,openalex_id,paperId,s2_url
0,openalex,https://openalex.org/W3202582446,dynamic pricing definition concepts brick-and-...,Digital Transformation: An Overview of the Cur...,2021.0,SAGE Open,article,Sascha Kraus; Paul Jones; Norbert Kailer; Alex...,https://doi.org/10.1177/21582440211047576,1044,The increasing digitalization of economies has...,https://openalex.org/W3202582446,NaN,NaN
1,openalex,https://openalex.org/W4313310852,dynamic pricing definition concepts brick-and-...,Metaverse marketing: How the metaverse will sh...,2022.0,Psychology and Marketing,article,Yogesh K. Dwivedi; Laurie Hughes; Yichuan Wang...,https://doi.org/10.1002/mar.21767,755,Abstract The initial hype and fanfare from the...,https://openalex.org/W4313310852,NaN,NaN
2,openalex,https://openalex.org/W2579462724,dynamic pricing definition concepts brick-and-...,Consumer-driven e-commerce,2018.0,International Journal of Physical Distribution...,article,Stanley Frederick W.T. Lim; Xin Jin; Jagjit Si...,https://doi.org/10.1108/ijpdlm-02-2017-0081,332,Purpose The purpose of this paper is to re-exa...,https://openalex.org/W2579462724,NaN,NaN
3,openalex,https://openalex.org/W191206311,dynamic pricing definition concepts brick-and-...,"The Evolution of Social Commerce: The People, ...",2012.0,Communications of the Association for Informat...,article,Chingning Wang; Ping Zhang,https://doi.org/10.17705/1cais.03105,533,Social commerce is a form of commerce mediated...,https://openalex.org/W191206311,NaN,NaN
4,openalex,https://openalex.org/W1548690256,dynamic pricing definition concepts brick-and-...,"Exploring Factors That Affect Usefulness, Ease...",2015.0,International Journal of Management & Informat...,article,Yoon C. Cho; Esen Sagynov,https://doi.org/10.19030/ijmis.v19i1.9086,299,Various studies have examined the effects of f...,https://openalex.org/W1548690256,NaN,NaN


In [31]:
def normalize_doi(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    if not x:
        return None
    x = re.sub(r"^https?://(dx\.)?doi\.org/", "", x)
    x = re.sub(r"^doi:\s*", "", x)
    x = x.strip()
    return x or None

def normalize_title(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    if not x:
        return None
    # remove punctuation-ish, collapse spaces
    x = re.sub(r"[\u2010-\u2015]", "-", x)   # normalize dash variants
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x or None


In [32]:
stageA = combined_raw.copy()

stageA["doi_norm"] = stageA["doi"].map(normalize_doi)
stageA["title_norm"] = stageA["title"].map(normalize_title)

# numeric citations
stageA["citation_count"] = pd.to_numeric(stageA["citation_count"], errors="coerce")

# keep only rows with a title (hard to use without)
stageA = stageA[stageA["title_norm"].notna()].reset_index(drop=True)

print("StageA rows (after title filter):", len(stageA))
stageA.head(5)


StageA rows (after title filter): 2124


,source,source_id,query,title,year,venue,type,authors(first6),doi,citation_count,abstract,openalex_id,paperId,s2_url,doi_norm,title_norm
0,openalex,https://openalex.org/W3202582446,dynamic pricing definition concepts brick-and-...,Digital Transformation: An Overview of the Cur...,2021.0,SAGE Open,article,Sascha Kraus; Paul Jones; Norbert Kailer; Alex...,https://doi.org/10.1177/21582440211047576,1044,The increasing digitalization of economies has...,https://openalex.org/W3202582446,NaN,NaN,10.1177/21582440211047576,digital transformation an overview of the curr...
1,openalex,https://openalex.org/W4313310852,dynamic pricing definition concepts brick-and-...,Metaverse marketing: How the metaverse will sh...,2022.0,Psychology and Marketing,article,Yogesh K. Dwivedi; Laurie Hughes; Yichuan Wang...,https://doi.org/10.1002/mar.21767,755,Abstract The initial hype and fanfare from the...,https://openalex.org/W4313310852,NaN,NaN,10.1002/mar.21767,metaverse marketing how the metaverse will sha...
2,openalex,https://openalex.org/W2579462724,dynamic pricing definition concepts brick-and-...,Consumer-driven e-commerce,2018.0,International Journal of Physical Distribution...,article,Stanley Frederick W.T. Lim; Xin Jin; Jagjit Si...,https://doi.org/10.1108/ijpdlm-02-2017-0081,332,Purpose The purpose of this paper is to re-exa...,https://openalex.org/W2579462724,NaN,NaN,10.1108/ijpdlm-02-2017-0081,consumer driven e commerce
3,openalex,https://openalex.org/W191206311,dynamic pricing definition concepts brick-and-...,"The Evolution of Social Commerce: The People, ...",2012.0,Communications of the Association for Informat...,article,Chingning Wang; Ping Zhang,https://doi.org/10.17705/1cais.03105,533,Social commerce is a form of commerce mediated...,https://openalex.org/W191206311,NaN,NaN,10.17705/1cais.03105,the evolution of social commerce the people ma...
4,openalex,https://openalex.org/W1548690256,dynamic pricing definition concepts brick-and-...,"Exploring Factors That Affect Usefulness, Ease...",2015.0,International Journal of Management & Informat...,article,Yoon C. Cho; Esen Sagynov,https://doi.org/10.19030/ijmis.v19i1.9086,299,Various studies have examined the effects of f...,https://openalex.org/W1548690256,NaN,NaN,10.19030/ijmis.v19i1.9086,exploring factors that affect usefulness ease ...


In [33]:
def longest_text(series):
    vals = [v for v in series if isinstance(v, str) and v.strip()]
    if not vals:
        return None
    return max(vals, key=len)

def most_common_or_longest(series):
    vals = [v for v in series if isinstance(v, str) and v.strip()]
    if not vals:
        return None
    vc = pd.Series(vals).value_counts()
    if len(vc) and vc.iloc[0] >= 2:
        return vc.index[0]
    return max(vals, key=len)

def first_nonnull(series):
    for v in series:
        if pd.notna(v) and v not in ("", None):
            return v
    return None

# Within-source grouping key: prefer DOI, else source_id, else title+year
def within_source_key(df):
    k = []
    for _, r in df.iterrows():
        if r["doi_norm"]:
            k.append(f"doi:{r['doi_norm']}")
        elif pd.notna(r["source_id"]):
            k.append(f"id:{r['source']}:{r['source_id']}")
        else:
            y = int(r["year"]) if pd.notna(r["year"]) else ""
            k.append(f"ty:{r['title_norm']}|{y}")
    return pd.Series(k, index=df.index)

stageA["within_key"] = within_source_key(stageA)

agg_map = {
    "source": lambda s: s.iloc[0],
    "source_id": first_nonnull,
    "title": most_common_or_longest,
    "title_norm": lambda s: s.iloc[0],
    "year": lambda s: pd.to_numeric(s, errors="coerce").dropna().median() if s.notna().any() else np.nan,
    "venue": most_common_or_longest,
    "type": most_common_or_longest,
    "authors(first6)": most_common_or_longest,
    "doi": first_nonnull,
    "doi_norm": first_nonnull,
    "citation_count": lambda s: pd.to_numeric(s, errors="coerce").max(),
    "abstract": longest_text,
    "query": lambda s: "; ".join(sorted(set([q for q in s if isinstance(q, str) and q.strip()])))
    ,
    "openalex_id": first_nonnull,
    "paperId": first_nonnull,
    "s2_url": first_nonnull,
}

stageA_dedup = (
    stageA
    .groupby(["source", "within_key"], as_index=False)
    .agg(agg_map)
    .drop(columns=["within_key"])
)

print("After within-source dedupe:", stageA_dedup.shape)
stageA_dedup.head(5)


After within-source dedupe: (2042, 16)


,source,source_id,title,title_norm,year,venue,type,authors(first6),doi,doi_norm,citation_count,abstract,query,openalex_id,paperId,s2_url
0,openalex,https://openalex.org/W2153932359,The impact of customer satisfaction and relati...,the impact of customer satisfaction and relati...,1997.0,Psychology and Marketing,article,Thorsten Hennig‐Thurau; Alexander Klee,https://doi.org/10.1002/(sici)1520-6793(199712...,10.1002/(sici)1520-6793(199712)14:8<737::aid-m...,1264,Customer satisfaction with a company's product...,dynamic pricing theory conceptual framework re...,https://openalex.org/W2153932359,None,None
1,openalex,https://openalex.org/W4238416262,The impact of customer satisfaction and relati...,the impact of customer satisfaction and relati...,1997.0,Psychology and Marketing,article,Thorsten Hennig‐Thurau; Alexander Klee,https://doi.org/10.1002/(sici)1520-6793(199712...,10.1002/(sici)1520-6793(199712)14:8<737::aid-m...,143,Customer satisfaction with a company's product...,dynamic pricing theory conceptual framework re...,https://openalex.org/W4238416262,None,None
2,openalex,https://openalex.org/W3097542903,Local Food Supply Chain Dynamics and Resilienc...,local food supply chain dynamics and resilienc...,2020.0,Applied Economic Perspectives and Policy,article,Dawn Thilmany; Elizabeth Canales; Sarah A. Low...,https://doi.org/10.1002/aepp.13121,10.1002/aepp.13121,252,Abstract Local and regional food systems (LRFS...,demand-based pricing demand-driven pricing dyn...,https://openalex.org/W3097542903,None,None
3,openalex,https://openalex.org/W2099487375,Australian wines in the British wine market: A...,australian wines in the british wine market a ...,2004.0,Agribusiness,article,Bodo Steiner,https://doi.org/10.1002/agr.20012,10.1002/agr.20012,106,Abstract The market share of New World wines s...,time-based pricing time-dependent pricing dyna...,https://openalex.org/W2099487375,None,None
4,openalex,https://openalex.org/W3203923637,Trends in Workplace Wearable Technologies and ...,trends in workplace wearable technologies and ...,2021.0,Advanced Intelligent Systems,article,Vishal Patel; Austin Chesmore; Christopher Leg...,https://doi.org/10.1002/aisy.202100099,10.1002/aisy.202100099,261,"The workplace influences the safety, health, a...",demand-based pricing demand-driven pricing dyn...,https://openalex.org/W3203923637,None,None


In [35]:
def cross_source_merge_key(r):
    if isinstance(r["doi_norm"], str) and r["doi_norm"]:
        return f"doi:{r['doi_norm']}"
    if pd.notna(r["year"]):
        return f"ty:{r['title_norm']}|{int(round(float(r['year'])))}"
    return f"t:{r['title_norm']}"

stageA_dedup["merge_key"] = stageA_dedup.apply(cross_source_merge_key, axis=1)

def merge_sources(group: pd.DataFrame) -> dict:
    # provenance
    sources = sorted(set(group["source"].dropna().tolist()))
    source_ids = {src: group.loc[group["source"] == src, "source_id"].dropna().astype(str).unique().tolist()
                  for src in sources}

    out = {
        "merge_key": group["merge_key"].iloc[0],
        "sources": "; ".join(sources),
        "source_count": len(sources),
        "source_ids": str(source_ids),
        "title": most_common_or_longest(group["title"]),
        "year": pd.to_numeric(group["year"], errors="coerce").dropna().median() if group["year"].notna().any() else np.nan,
        "venue": most_common_or_longest(group["venue"]),
        "type": most_common_or_longest(group["type"]),
        "authors(first6)": most_common_or_longest(group["authors(first6)"]),
        "doi": first_nonnull(group["doi"]),
        "doi_norm": first_nonnull(group["doi_norm"]),
        "citation_count_max": pd.to_numeric(group["citation_count"], errors="coerce").max(),
        "abstract": longest_text(group["abstract"]),
        "queries": "; ".join(sorted(set(
            q for q in group["query"].dropna().tolist()
            if isinstance(q, str) and q.strip()
        ))),
        # keep ids/urls if present
        "openalex_id": first_nonnull(group.get("openalex_id", pd.Series([], dtype=object))),
        "paperId": first_nonnull(group.get("paperId", pd.Series([], dtype=object))),
        "s2_url": first_nonnull(group.get("s2_url", pd.Series([], dtype=object))),
    }
    return out

merged_records = [merge_sources(g) for _, g in stageA_dedup.groupby("merge_key")]
df_stageA = pd.DataFrame(merged_records)

# Optional: sort by citation_count_max (just for inspection; ranking comes later)
df_stageA = df_stageA.sort_values(by="citation_count_max", ascending=False, na_position="last").reset_index(drop=True)

print("Stage A merged unique records:", df_stageA.shape)
df_stageA.head(10)


Stage A merged unique records: (2039, 17)


,merge_key,sources,source_count,source_ids,title,year,venue,type,authors(first6),doi,doi_norm,citation_count_max,abstract,queries,openalex_id,paperId,s2_url
0,doi:10.1007/s42979-021-00592-x,openalex,1,{'openalex': ['https://openalex.org/W313502870...,"Machine Learning: Algorithms, Real-World Appli...",2021.0,SN Computer Science,review,Iqbal H. Sarker,https://doi.org/10.1007/s42979-021-00592-x,10.1007/s42979-021-00592-x,4635,None,time-based pricing time-dependent pricing dyna...,https://openalex.org/W3135028703,None,None
1,doi:10.1111/j.1364-3703.2011.00783.x,openalex,1,{'openalex': ['https://openalex.org/W211875105...,The Top 10 fungal pathogens in molecular plant...,2012.0,Molecular Plant Pathology,review,Ralph A. Dean; J.A.L. van Kan; Z. A. Pretorius...,https://doi.org/10.1111/j.1364-3703.2011.00783.x,10.1111/j.1364-3703.2011.00783.x,4404,SUMMARY The aim of this review was to survey a...,time-based pricing time-dependent pricing dyna...,https://openalex.org/W2118751052,None,None
2,doi:10.1016/j.ijinfomgt.2019.08.002,openalex,1,{'openalex': ['https://openalex.org/W296962553...,Artificial Intelligence (AI): Multidisciplinar...,2019.0,International Journal of Information Management,article,Yogesh K. Dwivedi; Laurie Hughes; Elvira Ismag...,https://doi.org/10.1016/j.ijinfomgt.2019.08.002,10.1016/j.ijinfomgt.2019.08.002,3619,None,dynamic pricing theory conceptual framework re...,https://openalex.org/W2969625533,None,None
3,doi:10.25300/misq/2013/37:2.3,openalex,1,{'openalex': ['https://openalex.org/W214806016...,Digital Business Strategy: Toward a Next Gener...,2013.0,MIS Quarterly,article,Anandhi Bharadwaj; Omar A. El Sawy; Paul A. Pa...,https://doi.org/10.25300/misq/2013/37:2.3,10.25300/misq/2013/37:2.3,3569,"Over the last three decades, the prevailing vi...",dynamic pricing theory conceptual framework re...,https://openalex.org/W2148060162,None,None
4,doi:10.22004/ag.econ.288998,openalex,1,{'openalex': ['https://openalex.org/W218606665...,World agriculture towards 2030/2050: the 2012 ...,2012.0,RePEc: Research Papers in Economics,preprint,"N. Alexandratos; Jelle Bruinsma; Alexandratos,...",https://doi.org/10.22004/ag.econ.288998,10.22004/ag.econ.288998,3140,Current UN projections indicate that world pop...,dynamic pricing theory conceptual framework re...,https://openalex.org/W2186066651,None,None
5,doi:10.1016/s0140-6736(15)60901-1,openalex,1,{'openalex': ['https://openalex.org/W217575143...,Safeguarding human health in the Anthropocene ...,2015.0,The Lancet,review,Sarah Whitmee; Andy Haines; Chris Beyrer; Fred...,https://doi.org/10.1016/s0140-6736(15)60901-1,10.1016/s0140-6736(15)60901-1,2700,Earth's natural systems represent a growing th...,dynamic pricing theory conceptual framework re...,https://openalex.org/W2175751433,None,None
6,doi:10.1111/j.1749-6632.2009.05333.x,openalex,1,{'openalex': ['https://openalex.org/W213923603...,Neighborhoods and health,2010.0,Annals of the New York Academy of Sciences,article,Ana V. Diez Roux; Christina Mair,https://doi.org/10.1111/j.1749-6632.2009.05333.x,10.1111/j.1749-6632.2009.05333.x,2649,Features of neighborhoods or residential envir...,time-based pricing time-dependent pricing dyna...,https://openalex.org/W2139236032,None,None
7,doi:10.1257/aer.99.4.1145,openalex,1,{'openalex': ['https://openalex.org/W214554243...,Salience and Taxation: Theory and Evidence,2009.0,American Economic Review,article,Raj Chetty; Adam Looney; Kory Kroft,https://doi.org/10.1257/aer.99.4.1145,10.1257/aer.99.4.1145,2554,"Using two strategies, we show that consumers u...",time-based pricing time-dependent pricing dyna...,https://openalex.org/W2145542433,None,None
8,doi:10.1257/jel.52.3.740,openalex,1,{'openalex': ['https://openalex.org/W210483490...,What Do We Learn from the Weather? The New Cli...,2014.0,Journal of Economic Literature,article,Melissa Dell; Benjamin F. Jones; Benjamin Olken,https://doi.org/10.1257/jel.52.3.740,10.1257/jel.52.3.740,2089,A rapidly growing body of research applies pan...,dynamic

In [36]:
# How many are DOI-merged vs title-merged?
df_stageA["merge_kind"] = df_stageA["merge_key"].str.split(":", n=1).str[0]
df_stageA["has_abstract"] = df_stageA["abstract"].notna() & (df_stageA["abstract"].str.len() > 50)

print(df_stageA["merge_kind"].value_counts(dropna=False))
print("\nSources coverage:")
print(df_stageA["sources"].value_counts().head(10))

print("\nAbstract coverage:", df_stageA["has_abstract"].mean(), "(fraction with decent abstract)")

# Show examples where both sources matched
both = df_stageA[df_stageA["source_count"] >= 2].copy()
print("\nRecords with BOTH OpenAlex + S2:", len(both))
both.head(20)


merge_kind
doi    1806
ty      231
t         2
Name: count, dtype: int64

Sources coverage:
sources
openalex                      1620
semantic_scholar               417
openalex; semantic_scholar       2
Name: count, dtype: int64

Abstract coverage: 0.7788131436978911 (fraction with decent abstract)

Records with BOTH OpenAlex + S2: 2


,merge_key,sources,source_count,source_ids,title,year,venue,type,authors(first6),doi,doi_norm,citation_count_max,abstract,queries,openalex_id,paperId,s2_url,merge_kind,has_abstract
703,doi:10.2139/ssrn.2547793,openalex; semantic_scholar,2,{'openalex': ['https://openalex.org/W176637627...,Competition-Based Dynamic Pricing in Online Re...,2016.0,SSRN Electronic Journal,article,Marshall L. Fisher; Santiago Gallino; Jun Li,https://doi.org/10.2139/ssrn.2547793,10.2139/ssrn.2547793,154,A retailer following a competition-based dynam...,price discrimination dynamic pricing retail store,https://openalex.org/W1766376276,b9c5c004e5fa3a75972780d720f3bbf967d96fdc,https://www.semanticscholar.org/paper/b9c5c004...,doi,True
1591,doi:10.2139/ssrn.2994426,openalex; semantic_scholar,2,{'openalex': ['https://openalex.org/W274693232...,Dynamic Pricing and Organic Waste Bans: a Stud...,2018.5,SSRN Electronic Journal,article,Robert Evan Sanders,https://doi.org/10.2139/ssrn.2994426,10.2139/ssrn.2994426,5,None,demand-based pricing demand-driven pricing dyn...,https://openalex.org/W2746932329,75e46755c085c6e8e39d10d34b53b05675d492e1,https://www.semanticscholar.org/paper/75e46755...,doi,False


In [37]:
df_stageA.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV.resolve())

Saved: <projektverzeichnis>\stageA_combined_oa_s2.csv


In [38]:
# Use the within-source-deduped table from Cell 6: stageA_dedup
# (If you didn't keep it, reload from your earlier notebook state or rebuild quickly.)

def pct(x): 
    return round(100 * x, 2)

summary = (
    stageA_dedup.assign(
        has_doi = stageA_dedup["doi_norm"].notna(),
        has_year = stageA_dedup["year"].notna(),
        has_abs = stageA_dedup["abstract"].notna() & (stageA_dedup["abstract"].astype(str).str.len() > 50),
    )
    .groupby("source")[["has_doi","has_year","has_abs"]]
    .mean()
)

summary = summary.applymap(pct)
summary


<benutzerverzeichnis>\AppData\Local\Temp\ipykernel_8860\2123882543.py:17: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  summary = summary.applymap(pct)


,has_doi,has_year,has_abs
source,,,
openalex,87.85,100.00,80.64
semantic_scholar,91.19,99.52,66.67


In [39]:
oa = stageA_dedup[stageA_dedup["source"]=="openalex"].copy()
s2 = stageA_dedup[stageA_dedup["source"]=="semantic_scholar"].copy()

oa_dois = set(oa["doi_norm"].dropna().astype(str))
s2_dois = set(s2["doi_norm"].dropna().astype(str))

print("OpenAlex doi_norm count:", len(oa_dois))
print("S2 doi_norm count      :", len(s2_dois))
print("DOI intersection       :", len(oa_dois & s2_dois))

# Show a few intersecting DOIs if any
list(sorted(list(oa_dois & s2_dois)))[:20]


OpenAlex doi_norm count: 1425
S2 doi_norm count      : 383
DOI intersection       : 2


['10.2139/ssrn.2547793', '10.2139/ssrn.2994426']

In [40]:
def ty_key(df):
    d = df.copy()
    d["year_i"] = pd.to_numeric(d["year"], errors="coerce").round().astype("Int64")
    d["ty"] = np.where(d["year_i"].notna(), d["title_norm"] + "|" + d["year_i"].astype(str), None)
    return set(d["ty"].dropna().astype(str))

oa_ty = ty_key(oa)
s2_ty = ty_key(s2)

print("OpenAlex title|year keys:", len(oa_ty))
print("S2 title|year keys      :", len(s2_ty))
print("Intersection (title|yr) :", len(oa_ty & s2_ty))

list(sorted(list(oa_ty & s2_ty)))[:10]


OpenAlex title|year keys: 1612
S2 title|year keys      : 417
Intersection (title|yr) : 1


['an analysis of dynamic price discrimination in airlines|2018']

In [41]:
oa = stageA_dedup[stageA_dedup["source"]=="openalex"].copy()
s2 = stageA_dedup[stageA_dedup["source"]=="semantic_scholar"].copy()

oa_titles = set(oa["title_norm"].dropna().astype(str))
s2_titles = set(s2["title_norm"].dropna().astype(str))

print("OpenAlex title_norm:", len(oa_titles))
print("S2 title_norm      :", len(s2_titles))
print("Title intersection :", len(oa_titles & s2_titles))

# show a few exact-title matches (if any)
list(sorted(list(oa_titles & s2_titles)))[:20]


OpenAlex title_norm: 1605
S2 title_norm      : 417
Title intersection : 2


['an analysis of dynamic price discrimination in airlines',
 'competition based dynamic pricing in online retailing a methodology validated with field experiments']

---

# Stage B

---

In [43]:
import pandas as pd
from pathlib import Path

STAGEA_CSV = Path("stageA_combined_oa_s2.csv")  # <- your saved Stage A output
df_stageA = pd.read_csv(STAGEA_CSV)

# Keep only what we need for later ranking (title+abstract + a few metadata/tie-breakers)
cols_keep = [
    "merge_key","sources","title","year","venue","type","doi_norm",
    "citation_count_max","abstract","queries","openalex_id","paperId","s2_url"
]
df_stageA = df_stageA[[c for c in cols_keep if c in df_stageA.columns]].copy()

print("Loaded:", df_stageA.shape)
df_stageA.head(3)


Loaded: (2039, 13)


,merge_key,sources,title,year,venue,type,doi_norm,citation_count_max,abstract,queries,openalex_id,paperId,s2_url
0,doi:10.1007/s42979-021-00592-x,openalex,"Machine Learning: Algorithms, Real-World Appli...",2021.0,SN Computer Science,review,10.1007/s42979-021-00592-x,4635,NaN,time-based pricing time-dependent pricing dyna...,https://openalex.org/W3135028703,NaN,NaN
1,doi:10.1111/j.1364-3703.2011.00783.x,openalex,The Top 10 fungal pathogens in molecular plant...,2012.0,Molecular Plant Pathology,review,10.1111/j.1364-3703.2011.00783.x,4404,SUMMARY The aim of this review was to survey a...,time-based pricing time-dependent pricing dyna...,https://openalex.org/W2118751052,NaN,NaN
2,doi:10.1016/j.ijinfomgt.2019.08.002,openalex,Artificial Intelligence (AI): Multidisciplinar...,2019.0,International Journal of Information Management,article,10.1016/j.ijinfomgt.2019.08.002,3619,NaN,dynamic pricing theory conceptual framework re...,https://openalex.org/W2969625533,NaN,NaN


In [44]:
from pathlib import Path
import re

# Only change these inputs; everything downstream should adapt.
CHAPTER_ID = "2.1"  # used for filenames/cache keys
CHAPTER_TITLE = "Definition and concepts of Dynamic Pricing in brick-and-mortar retail"

# Chapter description can be in any language; set CHAPTER_TEXT_LANGUAGE accordingly.
CHAPTER_TEXT_DE = (
    "Die theoretischen Grundlagen des Dynamic Pricing werden detailliert beschrieben. "
    "Es wird erläutert, wie Dynamic Pricing im Einzelhandel funktioniert und welche "
    "unterschiedlichen Ansätze es gibt, z. B. zeitabhängige Preise oder an die Nachfrage "
    "gekoppelte Preisänderungen. Es wird nicht auf operative Details der Preisgestaltung "
    "oder spezifische Implementierungsstrategien eingegangen. Der Online-Handel wird "
    "explizit ausgeklammert."
)
CHAPTER_TEXT_LANGUAGE = "de"

# Optional hints (leave None/[] for fully automatic blueprinting)
CHAPTER_SCOPE_HINT_EN = None  # e.g., "Explain X and Y; exclude Z."
CHAPTER_MUST_COVER_HINTS = []
CHAPTER_MUST_AVOID_HINTS = []

# File naming
CHAPTER_SLUG = re.sub(r"[^0-9A-Za-z]+", "_", CHAPTER_ID).strip("_")
BLUEPRINT_JSON_PATH = Path(f"stageB_blueprint_{CHAPTER_SLUG}.json")

chapter_spec = {
    "chapter_id": CHAPTER_ID,
    "title": CHAPTER_TITLE,
    "original_text": CHAPTER_TEXT_DE,
    "original_text_language": CHAPTER_TEXT_LANGUAGE,
    "scope_hint_en": CHAPTER_SCOPE_HINT_EN,
    "must_cover_hints": CHAPTER_MUST_COVER_HINTS,
    "must_avoid_hints": CHAPTER_MUST_AVOID_HINTS,
}

chapter_spec


{'chapter_id': '2.1',
 'title': 'Definition and concepts of Dynamic Pricing in brick-and-mortar retail',
 'original_text': 'Die theoretischen Grundlagen des Dynamic Pricing werden detailliert beschrieben. Es wird erläutert, wie Dynamic Pricing im Einzelhandel funktioniert und welche unterschiedlichen Ansätze es gibt, z. B. zeitabhängige Preise oder an die Nachfrage gekoppelte Preisänderungen. Es wird nicht auf operative Details der Preisgestaltung oder spezifische Implementierungsstrategien eingegangen. Der Online-Handel wird explizit ausgeklammert.',
 'original_text_language': 'de',
 'scope_hint_en': None,
 'must_cover_hints': [],
 'must_avoid_hints': []}

In [45]:
from pydantic import BaseModel, Field
from typing import List, Optional

class ChapterBlueprint(BaseModel):
    chapter_id: str = Field(..., description="Chapter identifier (e.g., '2.1').")
    language: str = Field("en", description="Language of the queries.")
    
    # Rubric
    scope_statement: str = Field(..., description="Short English statement of the chapter goal.")
    must_cover: List[str] = Field(..., description="3–7 bullets: what a good source must cover.")
    should_cover: List[str] = Field(..., description="3–7 bullets: optional helpful coverage.")
    must_avoid: List[str] = Field(..., description="3–7 bullets: excluded scope / what should not dominate.")
    
    # Retrieval helpers
    main_query: str = Field(..., description="<= 18 words, broad query capturing the chapter scope.")
    facet_queries: List[str] = Field(..., description="6–12 narrower queries covering key sub-aspects.")
    
    keywords: List[str] = Field(..., description="15–40 keywords/phrases to help downstream matching.")
    key_concepts: List[str] = Field(..., description="8–20 conceptual phrases (more abstract than keywords).")

    preferred_source_types: Optional[List[str]] = Field(None, description="E.g., review, survey, textbook chapter, seminal paper.")
    negative_query_terms: Optional[List[str]] = Field(None, description="Soft-negative terms; NOT a hard filter.")
    
    scoring_guidance: str = Field(..., description="What high scores mean for relevance/conceptual_fit/scope_adherence.")
    notes: Optional[str] = Field(None, description="Short notes about intent/coverage limits.")

print("Schema ready.")


Schema ready.


In [46]:
import os
from agents import Agent, Runner, ModelSettings

# Ensure OPENAI_API_KEY is set in your environment before running.
# Example: export OPENAI_API_KEY="..."
assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY env var."

blueprint_agent = Agent(
    name="Chapter Blueprint Builder",
    model="gpt-5-nano",
    model_settings=ModelSettings(
        top_p=1.0,
        verbosity="low",
    ),
    instructions=(
        "You create a chapter blueprint (rubric) and search queries for academic literature retrieval.\n"
        "Return ONLY the structured output fields (no extra text).\n\n"
        "Constraints:\n"
        "- language must be 'en'\n"
        "- scope_statement: 1 sentence, <= 28 words\n"
        "- must_cover: 3–7 bullets, each <= 14 words\n"
        "- should_cover: 3–7 bullets, each <= 14 words\n"
        "- must_avoid: 3–7 bullets, each <= 14 words\n"
        "- main_query: <= 18 words, plain text, allow quotes for phrases\n"
        "- facet_queries: 6–12 items, each <= 14 words, plain text, allow quotes\n"
        "- keywords: 15–40 items (mix of single terms + short phrases)\n"
        "- key_concepts: 8–20 items (conceptual phrases; e.g., 'intertemporal price discrimination')\n"
        "- preferred_source_types: 2–6 items (strings)\n"
        "- negative_query_terms: 0–12 items; derived from must_avoid (NOT a hard filter)\n"
        "- scoring_guidance: <= 70 words\n"
        "- Do NOT hardcode any domain. Follow the given chapter spec.\n"
    ),
    output_type=ChapterBlueprint,
)

print("Agent ready.")


Agent ready.


In [47]:
import json
from pathlib import Path
from agents import Runner

payload = dict(chapter_spec)

prompt = (
    "Create a ChapterBlueprint for academic literature retrieval.\n"
    "Return ONLY the structured output fields required by the schema.\n\n"
    "CHAPTER_SPEC_JSON:\n"
    f"{json.dumps(payload, ensure_ascii=False, indent=2)}"
)

# Jupyter: use `await` (NOT run_sync, NOT asyncio.run)
result = await Runner.run(blueprint_agent, prompt)

blueprint = result.final_output
blueprint_dict = blueprint.model_dump()

BLUEPRINT_JSON_PATH.write_text(
    json.dumps(blueprint_dict, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
print("Saved:", BLUEPRINT_JSON_PATH)


Saved: stageB_blueprint_2_1.json


AttributeError: 'ChapterBlueprint' object has no attribute 'get'

In [49]:
import pandas as pd

df_must = pd.DataFrame({"must_cover": blueprint_dict["must_cover"]})
df_should = pd.DataFrame({"should_cover": blueprint_dict["should_cover"]})
df_avoid = pd.DataFrame({"must_avoid": blueprint_dict["must_avoid"]})
df_facets = pd.DataFrame({"facet_queries": blueprint_dict["facet_queries"]})
df_keywords = pd.DataFrame({"keywords": blueprint_dict["keywords"]})
df_concepts = pd.DataFrame({"key_concepts": blueprint_dict["key_concepts"]})

print("Scope statement:\n", blueprint_dict["scope_statement"])
print("Main query:\n", blueprint_dict["main_query"])
print("Preferred source types:", blueprint_dict.get("preferred_source_types"))
print("Negative query terms:", blueprint_dict.get("negative_query_terms"))
display(df_must)
display(df_should)
display(df_avoid)
display(df_facets.head(30))
display(df_keywords.head(60))
display(df_concepts.head(60))


Scope statement:
 The chapter surveys dynamic pricing theory in brick-and-mortar retail, detailing pricing approaches and their conceptual foundations, excluding online channels.
Main query:
 What are the theoretical foundations and mainstream approaches of dynamic pricing in brick-and-mortar retail?
Preferred source types: ['review article', 'survey', 'textbook chapter', 'academic journal article']
Negative query terms: ['online retail', 'e-commerce', 'online channels', 'operational pricing details', 'real-time pricing online', 'online promotions']


,must_cover
0,Theoretical foundations of dynamic pricing in ...
1,Different pricing approaches: time-based and d...
2,Context: brick-and-mortar retail; online chann...
3,Conceptual distinctions between price optimiza...
4,Non-operational scope: omit implementation spe...


,should_cover
0,Theoretical vs empirical evidence in offline p...
1,Impact of temporal variations on demand.
2,Price signaling without operational detail.
3,Regulatory and ethical considerations in pricing.
4,Measurement challenges: elasticity and cross-e...
5,Theoretical differences between promotions and...


,must_avoid
0,Online commerce and e-commerce dynamics.
1,Operational pricing details and real-time algo...
2,Case studies of online channels.
3,Excludes tactical promotions and loyalty effects.


,facet_queries
0,time-based pricing
1,demand-based pricing
2,price discrimination in retail
3,offline versus online pricing differences
4,consumer response to price changes
5,data requirements for price setting
6,seasonality and promotions effects
7,pricing signaling and competition spillovers


,keywords
0,dynamic pricing
1,brick-and-mortar
2,retail pricing
3,price elasticity
4,price optimization
5,demand-based pricing
6,time-based pricing
7,price discrimination
8,price signaling
9,consumer behavior


,key_concepts
0,dynamic pricing theory
1,brick-and-mortar context
2,time-based adjustment
3,demand-based adjustment
4,price discrimination in retail
5,price signaling
6,elasticity of demand
7,consumer surplus implications
8,competitive spillovers
9,price transparency


In [50]:
def contains_any(text, keywords):
    if not isinstance(text, str):
        return False
    t = text.lower()
    return any(k in t for k in keywords)

# Optional coverage diagnostic: how many docs mention chapter terms.
# (Run Stage B first for an automatic keyword list.)
import re
import pandas as pd

if "blueprint" in globals():
    keywords = [str(k).lower() for k in (blueprint.get("keywords") or [])][:25]
    col = "has_blueprint_terms"
elif "CHAPTER_TITLE" in globals():
    keywords = [w.lower() for w in re.findall(r"[A-Za-zÄÖÜäöüß]{4,}", str(CHAPTER_TITLE))][:25]
    col = "has_title_terms"
else:
    keywords = []
    col = "has_terms"

tmp = stageA_dedup.copy()
tmp["text"] = (tmp["title"].fillna("") + " " + tmp["abstract"].fillna("")).astype(str)

if not keywords:
    print("No keywords available. Run Stage B to generate a blueprint, then re-run this cell.")
else:
    tmp[col] = tmp["text"].apply(lambda x: contains_any(x, keywords))
    display(pd.DataFrame({"keyword": keywords}))
    tmp.groupby("source")[col].mean().mul(100).round(2)


AttributeError: 'ChapterBlueprint' object has no attribute 'get'

---

# Stage C

---

## C.1

In [51]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import re

STAGEA_CSV = Path("stageA_combined_oa_s2.csv")

# Uses the blueprint generated in Stage B.
if "BLUEPRINT_JSON_PATH" in globals():
    BLUEPRINT_JSON = BLUEPRINT_JSON_PATH
else:
    if "CHAPTER_ID" not in globals():
        raise RuntimeError("Missing BLUEPRINT_JSON_PATH/CHAPTER_ID. Run Stage B first.")
    chapter_slug = re.sub(r"[^0-9A-Za-z]+", "_", CHAPTER_ID).strip("_")
    BLUEPRINT_JSON = Path(f"stageB_blueprint_{chapter_slug}.json")

df = pd.read_csv(STAGEA_CSV)
blueprint = json.loads(BLUEPRINT_JSON.read_text(encoding="utf-8"))

print("StageA:", df.shape)
print("Blueprint keys:", list(blueprint.keys()))
df.head(1)


StageA: (2039, 19)
Blueprint keys: ['chapter_id', 'language', 'scope_statement', 'must_cover', 'should_cover', 'must_avoid', 'main_query', 'facet_queries', 'keywords', 'key_concepts', 'preferred_source_types', 'negative_query_terms', 'scoring_guidance', 'notes']


,merge_key,sources,source_count,source_ids,title,year,venue,type,authors(first6),doi,doi_norm,citation_count_max,abstract,queries,openalex_id,paperId,s2_url,merge_kind,has_abstract
0,doi:10.1007/s42979-021-00592-x,openalex,1,{'openalex': ['https://openalex.org/W313502870...,"Machine Learning: Algorithms, Real-World Appli...",2021.0,SN Computer Science,review,Iqbal H. Sarker,https://doi.org/10.1007/s42979-021-00592-x,10.1007/s42979-021-00592-x,4635,NaN,time-based pricing time-dependent pricing dyna...,https://openalex.org/W3135028703,NaN,NaN,doi,False


In [52]:
def build_query_text(blueprint: dict) -> str:
    parts = []
    # Give the main query more weight by repeating it
    main = blueprint["main_query"]
    parts.extend([main, main])

    # Facets: medium weight
    for fq in blueprint.get("facet_queries", []):
        parts.append(fq)

    # Keywords + concepts: lighter weight (but still useful)
    parts.extend(blueprint.get("keywords", []))
    parts.extend(blueprint.get("key_concepts", []))

    return " ".join(parts)

chapter_query_text = build_query_text(blueprint)
print(chapter_query_text[:600] + " ...")


What are the theoretical foundations and mainstream approaches of dynamic pricing in brick-and-mortar retail? What are the theoretical foundations and mainstream approaches of dynamic pricing in brick-and-mortar retail? time-based pricing demand-based pricing price discrimination in retail offline versus online pricing differences consumer response to price changes data requirements for price setting seasonality and promotions effects pricing signaling and competition spillovers dynamic pricing brick-and-mortar retail pricing price elasticity price optimization demand-based pricing time-based  ...


In [53]:
def safe_str(x):
    return "" if pd.isna(x) else str(x)

df = df.copy()
df["title"] = df["title"].fillna("")
df["abstract"] = df["abstract"].fillna("")
df["doc_text"] = (df["title"].astype(str) + "\n\n" + df["abstract"].astype(str)).str.strip()

# Basic stats
df["has_abstract"] = df["abstract"].str.len() > 50
print("Docs:", len(df))
print("Abstract coverage:", df["has_abstract"].mean())
df[["title","has_abstract","citation_count_max"]].head(5)


Docs: 2039
Abstract coverage: 0.7788131436978911


,title,has_abstract,citation_count_max
0,"Machine Learning: Algorithms, Real-World Appli...",False,4635
1,The Top 10 fungal pathogens in molecular plant...,True,4404
2,Artificial Intelligence (AI): Multidisciplinar...,False,3619
3,Digital Business Strategy: Toward a Next Gener...,True,3569
4,World agriculture towards 2030/2050: the 2012 ...,True,3140


In [54]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Keep it efficient: cap vocab size; include bigrams for phrases like "brick and mortar"
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=200_000,
    min_df=2
)

X = vectorizer.fit_transform(df["doc_text"])
q = vectorizer.transform([chapter_query_text])

tfidf_score = cosine_similarity(q, X).ravel()
df["score_tfidf"] = tfidf_score

df[["title","score_tfidf"]].sort_values("score_tfidf", ascending=False).head(10)


,title,score_tfidf
567,Price Discrimination in E-Commerce? An Examina...,0.337048
1765,When should grocery stores adopt time‐based pr...,0.297231
1349,Exploring customers’ likeliness to use e-servi...,0.279728
2031,IMPACT OF DYNAMIC PRICING ON PROFIT MARGINS,0.266306
1830,Demand-based pricing versus past-price depende...,0.259246
391,Price Discrimination and Retail Configuration,0.239305
1921,COMPARING APPROACHES: A SCIENTIFIC OVERVIEW OF...,0.235945
1491,Leveraging Microservices Architecture for Dyna...,0.234654
1916,© Generative AI-powered dynamic pricing in e-c...,0.232423
980,Personalized pricing and price fairness,0.230875


In [55]:
# If missing, treat as 0
cites = pd.to_numeric(df.get("citation_count_max", 0), errors="coerce").fillna(0).clip(lower=0)

# log1p smooths extremes: 0->0, 10->~2.4, 100->~4.6, 1000->~6.9
df["score_cite"] = np.log1p(cites)

# Normalize cite score to 0..1 for stable weighting
if df["score_cite"].max() > 0:
    df["score_cite_norm"] = df["score_cite"] / df["score_cite"].max()
else:
    df["score_cite_norm"] = 0.0

# Final baseline score: mostly relevance, tiny citation bump
CITE_WEIGHT = 0.08
df["score_stageC1"] = df["score_tfidf"] * (1 - CITE_WEIGHT) + df["score_cite_norm"] * CITE_WEIGHT

df[["title","score_tfidf","score_cite_norm","score_stageC1"]].sort_values("score_stageC1", ascending=False).head(10)


,title,score_tfidf,score_cite_norm,score_stageC1
567,Price Discrimination in E-Commerce? An Examina...,0.337048,0.628234,0.360343
1349,Exploring customers’ likeliness to use e-servi...,0.279728,0.381311,0.287854
1765,When should grocery stores adopt time‐based pr...,0.297231,0.130143,0.283864
391,Price Discrimination and Retail Configuration,0.239305,0.667502,0.273561
980,Personalized pricing and price fairness,0.230875,0.534360,0.255154
963,The forgotten sons: Warehousing systems for br...,0.225213,0.538203,0.250252
1830,Demand-based pricing versus past-price depende...,0.259246,0.082111,0.245075
2031,IMPACT OF DYNAMIC PRICING ON PROFIT MARGINS,0.266306,0.000000,0.245002
1491,Leveraging Microservices Architecture for Dyna...,0.234654,0.284057,0.238606
1204,Graph Deep-Learning-Based Retail Dynamic Prici...,0.219747,0.448278,0.238030


In [56]:
TOP_N = 50

cols_show = [
    "score_stageC1","score_tfidf","score_cite_norm",
    "title","year","venue","sources","doi_norm","citation_count_max","has_abstract"
]

top = df.sort_values("score_stageC1", ascending=False).head(TOP_N)[cols_show].reset_index(drop=True)
top


,score_stageC1,score_tfidf,score_cite_norm,title,year,venue,sources,doi_norm,citation_count_max,has_abstract
0,0.360343,0.337048,0.628234,Price Discrimination in E-Commerce? An Examina...,2011.0,MIS Q.,semantic_scholar,10.2307/23043490,200,False
1,0.287854,0.279728,0.381311,Exploring customers’ likeliness to use e-servi...,2020.0,Electronic Markets,openalex,10.1007/s12525-020-00445-0,24,False
2,0.283864,0.297231,0.130143,When should grocery stores adopt time‐based pr...,2023.0,Production and operations management,semantic_scholar,10.1111/poms.14010,2,True
3,0.273561,0.239305,0.667502,Price Discrimination and Retail Configuration,1991.0,Journal of Political Economy,semantic_scholar,10.1086/261739,279,False
4,0.255154,0.230875,0.534360,Personalized pricing and price fairness,2015.0,International Journal of Industrial Organization,openalex,10.1016/j.ijindorg.2015.11.004,90,False
5,0.250252,0.225213,0.538203,The forgotten sons: Warehousing systems for br...,2020.0,European Journal of Operational Research,openalex,10.1016/j.ejor.2020.04.058,93,False
6,0.245075,0.259246,0.082111,Demand-based pricing versus past-price depende...,2006.0,NaN,semantic_scholar,NaN,1,False
7,0.245002,0.266306,0.000000,IMPACT OF DYNAMIC PRICING ON PROFIT MARGINS,2025.0,EPRA International Journal of Economic and Bus...,semantic_scholar,10.36713/epra21984,0,True
8,0.238606,0.234654,0.284057,Leveraging Microservices Architecture for Dyna...,2024.0,arXiv.org,semantic_scholar,10.48550/arxiv.2411.01636,10,True
9,0.238030,0.219747,0.448278,Graph Deep-Learning-Based Retail Dynamic Prici...,2023.0,IEEE Transactions on Smart Grid,semantic_scholar,10.1109/tsg.2023.3258605,43,True


## C.2

In [57]:
import numpy as np
import pandas as pd

# We assume you already have:
# - df with columns: merge_key, title, abstract, doc_text, score_tfidf, score_cite_norm
# - vectorizer, X already computed from Stage C.1 (TF-IDF matrix for df["doc_text"])
# If you don't, re-run Stage C.1 cells (vectorizer + X + score_tfidf).

from sklearn.metrics.pairwise import cosine_similarity

# Use main + facets as separate "retrieval queries" (repeat main for extra weight downstream)
query_texts = [blueprint["main_query"], blueprint["main_query"]] + blueprint.get("facet_queries", [])

TOP_PER_QUERY = 250  # tune: 150–400 are common; larger = more embedding cost

# Compute TF-IDF similarity per query and union top ids
pool_keys = set()
for qt in query_texts:
    qv = vectorizer.transform([qt])
    sims = cosine_similarity(qv, X).ravel()
    top_idx = np.argsort(-sims)[:TOP_PER_QUERY]
    pool_keys.update(df.iloc[top_idx]["merge_key"].astype(str).tolist())

df_pool = df[df["merge_key"].astype(str).isin(pool_keys)].copy()
df_pool = df_pool.drop_duplicates(subset=["merge_key"]).reset_index(drop=True)

print("Facet-union embedding pool size:", len(df_pool))
df_pool[["title","score_tfidf","score_stageC1"]].head(5)


Facet-union embedding pool size: 930


,title,score_tfidf,score_stageC1
0,The Top 10 fungal pathogens in molecular plant...,0.003225,0.082482
1,Artificial Intelligence (AI): Multidisciplinar...,0.000000,0.077656
2,Digital Business Strategy: Toward a Next Gener...,0.002955,0.080243
3,World agriculture towards 2030/2050: the 2012 ...,0.007397,0.083116
4,Safeguarding human health in the Anthropocene ...,0.008418,0.082625


In [58]:
from pathlib import Path

MAX_CHARS_PER_DOC = 3500
BATCH_SIZE = 64
EMBED_MODEL = "text-embedding-3-small"

CACHE_DIR = Path(".embed_cache")
CACHE_DIR.mkdir(exist_ok=True)

POOL_TAG = f"facetUnion_q{len(query_texts)}_top{TOP_PER_QUERY}_n{len(df_pool)}"
DOC_EMBED_NPZ = CACHE_DIR / f"doc_embeds_{EMBED_MODEL}_{POOL_TAG}.npz"
QUERY_EMBED_JSON = CACHE_DIR / f"query_embeds_{EMBED_MODEL}_{POOL_TAG}.json"

def truncate_text(s: str, max_chars: int) -> str:
    s = "" if s is None else str(s)
    s = s.replace("\n", " ").strip()
    return s[:max_chars]

df_pool["doc_text_trunc"] = (df_pool["title"].fillna("") + "\n\n" + df_pool["abstract"].fillna("")).apply(
    lambda x: truncate_text(x, MAX_CHARS_PER_DOC)
)

print("Pool ready:", df_pool.shape)


Pool ready: (930, 25)


In [59]:
import os, json, time
import numpy as np
from openai import OpenAI

client = OpenAI()
assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY env var."

def embed_texts(texts, model=EMBED_MODEL, batch_size=BATCH_SIZE, max_retries=6):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        backoff = 1.0
        for attempt in range(1, max_retries + 1):
            try:
                resp = client.embeddings.create(model=model, input=batch)
                vecs = [np.array(item.embedding, dtype=np.float32) for item in resp.data]
                all_vecs.extend(vecs)
                break
            except Exception:
                if attempt == max_retries:
                    raise
                time.sleep(backoff)
                backoff *= 2
    return np.vstack(all_vecs)

def save_npz(path, keys, mat):
    np.savez_compressed(path, keys=np.array(keys, dtype=object), embeds=mat.astype(np.float32))

def load_npz(path):
    z = np.load(path, allow_pickle=True)
    return list(z["keys"]), z["embeds"].astype(np.float32)

# ---- doc embeddings
if DOC_EMBED_NPZ.exists():
    cached_keys, doc_embeds = load_npz(DOC_EMBED_NPZ)
    print("Loaded cached doc embeddings:", doc_embeds.shape)
else:
    cached_keys = df_pool["merge_key"].astype(str).tolist()
    doc_embeds = embed_texts(df_pool["doc_text_trunc"].tolist())
    save_npz(DOC_EMBED_NPZ, cached_keys, doc_embeds)
    print("Computed+saved doc embeddings:", doc_embeds.shape)

key_to_i = {k: i for i, k in enumerate(cached_keys)}

# ---- query embeddings
if QUERY_EMBED_JSON.exists():
    q_cached = json.loads(QUERY_EMBED_JSON.read_text(encoding="utf-8"))
    query_embeds = np.array(q_cached["embeddings"], dtype=np.float32)
    print("Loaded cached query embeddings:", query_embeds.shape)
else:
    query_embeds = embed_texts(query_texts)
    QUERY_EMBED_JSON.write_text(json.dumps({
        "model": EMBED_MODEL,
        "query_texts": query_texts,
        "embeddings": query_embeds.tolist(),
    }, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Computed+saved query embeddings:", query_embeds.shape)


Computed+saved doc embeddings: (930, 1536)
Computed+saved query embeddings: (10, 1536)


In [60]:
def l2_normalize(mat: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12
    return mat / norms

def minmax(x):
    x = np.asarray(x, dtype=float)
    mn, mx = np.nanmin(x), np.nanmax(x)
    return (x - mn) / (mx - mn + 1e-12)

docN = l2_normalize(doc_embeds)
qN = l2_normalize(query_embeds)

S = docN @ qN.T
score_embed_max = S.max(axis=1)
score_embed_mean_top3 = np.sort(S, axis=1)[:, -3:].mean(axis=1)

df_pool = df_pool.copy()
df_pool["score_embed_max"] = df_pool["merge_key"].astype(str).map(lambda k: score_embed_max[key_to_i[k]])
df_pool["score_embed_mean_top3"] = df_pool["merge_key"].astype(str).map(lambda k: score_embed_mean_top3[key_to_i[k]])

df_pool["score_embed_norm"] = minmax(df_pool["score_embed_max"].values)
df_pool["score_embed_mean_top3_norm"] = minmax(df_pool["score_embed_mean_top3"].values)

# Blend best-hit + breadth (main query was repeated upstream to add weight here)
W_EMBED_MAX = 0.70
W_EMBED_BREADTH = 0.30
df_pool["score_embed_combo"] = W_EMBED_MAX * df_pool["score_embed_norm"] + W_EMBED_BREADTH * df_pool["score_embed_mean_top3_norm"]

# normalize tfidf within pool for fair fusion
df_pool["score_tfidf_norm"] = minmax(df_pool["score_tfidf"].values)

W_EMBED = 0.60
W_TFIDF = 0.40
df_pool["score_relevance_hybrid"] = W_EMBED * df_pool["score_embed_combo"] + W_TFIDF * df_pool["score_tfidf_norm"]

CITE_WEIGHT = 0.08
df_pool["score_hybrid_pool"] = (1 - CITE_WEIGHT) * df_pool["score_relevance_hybrid"] + CITE_WEIGHT * df_pool["score_cite_norm"]

df_pool.sort_values("score_hybrid_pool", ascending=False).head(10)[
    ["title","score_embed_max","score_embed_mean_top3","score_tfidf","score_hybrid_pool","score_stageC1"]
]


,title,score_embed_max,score_embed_mean_top3,score_tfidf,score_hybrid_pool,score_stageC1
174,Price Discrimination in E-Commerce? An Examina...,0.658239,0.601863,0.337048,0.870631,0.360343
122,Price Discrimination and Retail Configuration,0.744830,0.589354,0.239305,0.812287,0.273561
605,Price Discrimination in Online Retail,0.800867,0.636051,0.175797,0.754531,0.181440
923,IMPACT OF DYNAMIC PRICING ON PROFIT MARGINS,0.632233,0.587276,0.266306,0.724191,0.245002
502,A retail benchmarking approach to efficient tw...,0.658428,0.597071,0.206912,0.706707,0.220073
739,When should grocery stores adopt time‐based pr...,0.550251,0.530057,0.297231,0.705365,0.283864
787,Demand-based pricing versus past-price depende...,0.604939,0.579111,0.259246,0.705255,0.245075
844,COMPARING APPROACHES: A SCIENTIFIC OVERVIEW OF...,0.633386,0.605872,0.235945,0.697123,0.217069
308,Personalized pricing and price fairness,0.579197,0.555445,0.230875,0.689012,0.255154
699,DYNAMIC PRICING: THE FUTURE OF RETAIL BUSINESS,0.672432,0.628738,0.189523,0.688301,0.187499


In [61]:
TOP_FOR_LLM = 150
MAX_ABS_CHARS_FOR_LLM = 1200  # keep costs low

df_llm = (
    df_pool.sort_values("score_hybrid_pool", ascending=False)
    .head(TOP_FOR_LLM)
    .copy()
    .reset_index(drop=True)
)

def llm_doc_text(title, abstract, max_chars=MAX_ABS_CHARS_FOR_LLM):
    t = ("" if pd.isna(title) else str(title)).strip()
    a = ("" if pd.isna(abstract) else str(abstract)).strip().replace("\n", " ")
    a = a[:max_chars]
    return f"TITLE: {t}\nABSTRACT: {a}"

df_llm["llm_text"] = df_llm.apply(lambda r: llm_doc_text(r["title"], r["abstract"]), axis=1)

print("LLM rerank set:", df_llm.shape)
df_llm[["title","score_hybrid_pool"]].head(10)


LLM rerank set: (150, 34)


,title,score_hybrid_pool
0,Price Discrimination in E-Commerce? An Examina...,0.870631
1,Price Discrimination and Retail Configuration,0.812287
2,Price Discrimination in Online Retail,0.754531
3,IMPACT OF DYNAMIC PRICING ON PROFIT MARGINS,0.724191
4,A retail benchmarking approach to efficient tw...,0.706707
5,When should grocery stores adopt time‐based pr...,0.705365
6,Demand-based pricing versus past-price depende...,0.705255
7,COMPARING APPROACHES: A SCIENTIFIC OVERVIEW OF...,0.697123
8,Personalized pricing and price fairness,0.689012
9,DYNAMIC PRICING: THE FUTURE OF RETAIL BUSINESS,0.688301


In [62]:
from pydantic import BaseModel, Field
from agents import Agent, ModelSettings
from typing import List, Optional

class RerankScore(BaseModel):
    relevance: int = Field(..., ge=0, le=100, description="Overall relevance to chapter scope (0-100).")
    conceptual_fit: int = Field(..., ge=0, le=100, description="Match to desired conceptual vs implementation focus (0-100).")
    scope_adherence: int = Field(..., ge=0, le=100, description="Respects must_cover/must_avoid from the chapter blueprint (0-100).")
    facet_hits: Optional[List[int]] = Field(None, description="Optional facet indices (0-based) that the source strongly matches.")
    notes: str = Field(..., description="Very short reason (<= 18 words).")

# Bump this if you change instructions to avoid reusing old cache accidentally
INSTRUCTIONS_VERSION = "v3_blueprint_rubric"

rerank_agent = Agent(
    name="Chapter Reranker",
    model="gpt-5-nano",
    model_settings=ModelSettings(verbosity="low"),
    instructions=(
        "You are reranking candidate sources for writing a thesis chapter.\n\n"
        "You will receive a CHAPTER_BLUEPRINT and a CANDIDATE (title+abstract).\n"
        "Score the candidate according to the blueprint.\n\n"
        "SCORING SCALE (MANDATORY):\n"
        "- Use INTEGER scores from 0 to 100 (NOT 0-10).\n"
        "- 0 = irrelevant\n"
        "- 50 = somewhat relevant\n"
        "- 80 = strong fit\n"
        "- 100 = perfect fit\n"
        "- Use the full range when appropriate.\n\n"
        "OUTPUT FIELDS:\n"
        "- relevance (0-100)\n"
        "- conceptual_fit (0-100)\n"
        "- scope_adherence (0-100)\n"
        "- facet_hits: optional list of up to 3 facet indices (0-based)\n"
        "- notes: <= 18 words\n\n"
        "Return ONLY the structured output.\n"
    ),
    output_type=RerankScore,
)


In [63]:
import asyncio
import time
import json
import hashlib
from pathlib import Path
import pandas as pd

from agents import Runner
from tqdm.auto import tqdm

# Pricing for gpt-5-nano (per 1M tokens)
PRICE_INPUT_PER_1M  = 0.05
PRICE_CACHED_PER_1M = 0.005
PRICE_OUTPUT_PER_1M = 0.40

# Cache directory (new versioned folder to avoid mixing old outputs)
LLM_CACHE = Path(f".llm_rerank_cache_{INSTRUCTIONS_VERSION}")
LLM_CACHE.mkdir(exist_ok=True)

def cost_from_usage(usage) -> dict:
    """
    Computes token totals + cost from Agents SDK usage.
    Uses per-request entries when available; falls back to aggregated totals.
    """
    def req_cost(input_tokens, cached_tokens, output_tokens):
        cached_tokens = int(cached_tokens or 0)
        input_tokens  = int(input_tokens or 0)
        output_tokens = int(output_tokens or 0)

        non_cached = max(0, input_tokens - cached_tokens)

        cost = (
            (non_cached    / 1_000_000) * PRICE_INPUT_PER_1M +
            (cached_tokens / 1_000_000) * PRICE_CACHED_PER_1M +
            (output_tokens / 1_000_000) * PRICE_OUTPUT_PER_1M
        )
        return non_cached, cached_tokens, output_tokens, cost

    entries = getattr(usage, "request_usage_entries", None) or []
    if entries:
        total_in = total_cached = total_out = 0
        total_cost = 0.0
        for r in entries:
            inp = getattr(r, "input_tokens", 0) or 0
            out = getattr(r, "output_tokens", 0) or 0
            itd = getattr(r, "input_tokens_details", None)
            cached = getattr(itd, "cached_tokens", 0) if itd is not None else 0
            non_cached, cached, out, c = req_cost(inp, cached, out)
            total_in += non_cached
            total_cached += cached
            total_out += out
            total_cost += c
        return {
            "requests": int(getattr(usage, "requests", len(entries)) or len(entries)),
            "input_tokens": int(total_in + total_cached),
            "cached_input_tokens": int(total_cached),
            "output_tokens": int(total_out),
            "cost_usd": float(total_cost),
        }

    # Fallback: aggregated totals
    inp = int(getattr(usage, "input_tokens", 0) or 0)
    out = int(getattr(usage, "output_tokens", 0) or 0)
    itd = getattr(usage, "input_tokens_details", None)
    cached = int(getattr(itd, "cached_tokens", 0) if itd is not None else 0)

    non_cached, cached, out, c = req_cost(inp, cached, out)
    return {
        "requests": int(getattr(usage, "requests", 1) or 1),
        "input_tokens": int(non_cached + cached),
        "cached_input_tokens": int(cached),
        "output_tokens": int(out),
        "cost_usd": float(c),
    }

def fmt_bullets(items):
    items = items or []
    return "\n".join(f"- {x}" for x in items) if items else "- (none)"

def fmt_facets(facet_queries):
    facet_queries = facet_queries or []
    return "\n".join(f"{i}. {q}" for i, q in enumerate(facet_queries)) if facet_queries else "(none)"

def build_prompt(blueprint: dict, llm_text: str) -> str:
    return (
        "CHAPTER_BLUEPRINT:\n"
        f"SCOPE_STATEMENT: {blueprint.get('scope_statement','')}\n"
        f"MAIN_QUERY: {blueprint.get('main_query','')}\n"
        "MUST_COVER:\n"
        f"{fmt_bullets(blueprint.get('must_cover'))}\n"
        "SHOULD_COVER:\n"
        f"{fmt_bullets(blueprint.get('should_cover'))}\n"
        "MUST_AVOID:\n"
        f"{fmt_bullets(blueprint.get('must_avoid'))}\n"
        f"PREFERRED_SOURCE_TYPES: {', '.join(blueprint.get('preferred_source_types') or [])}\n"
        f"NEGATIVE_QUERY_TERMS: {', '.join(blueprint.get('negative_query_terms') or [])}\n"
        "FACET_QUERIES (0-based indices):\n"
        f"{fmt_facets(blueprint.get('facet_queries'))}\n"
        f"SCORING_GUIDANCE: {blueprint.get('scoring_guidance','')}\n"
        f"NOTES: {blueprint.get('notes','')}\n\n"
        "CANDIDATE:\n"
        f"{llm_text}\n"
    )

def cache_path_for_prompt(prompt: str, model: str, instructions_version: str) -> Path:
    """
    Robust cache key: model + instructions_version + full prompt.
    Prevents reusing stale outputs across chapters/instruction changes.
    """
    blob = model + "\n" + instructions_version + "\n" + prompt
    h = hashlib.sha1(blob.encode("utf-8")).hexdigest()
    return LLM_CACHE / f"{h}.json"

async def rerank_one(idx: int, row, max_retries: int = 6):
    """
    Returns: (idx, scores_dict, usage_dict)
    If loaded from local cache: usage_dict is zeros.
    """
    llm_text = row["llm_text"]
    prompt = build_prompt(blueprint, llm_text)
    cp = cache_path_for_prompt(prompt, model="gpt-5-nano", instructions_version=INSTRUCTIONS_VERSION)

    # Local cache hit => no API call
    if cp.exists():
        out = json.loads(cp.read_text(encoding="utf-8"))
        usage0 = {"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}
        return idx, out, usage0

    backoff = 1.0
    for attempt in range(1, max_retries + 1):
        try:
            res = await Runner.run(rerank_agent, prompt)
            out = res.final_output.model_dump()
            cp.write_text(json.dumps(out, ensure_ascii=False), encoding="utf-8")

            usage = res.context_wrapper.usage
            usage_dict = cost_from_usage(usage)
            return idx, out, usage_dict
        except Exception:
            if attempt == max_retries:
                raise
            await asyncio.sleep(backoff + 0.15 * attempt)
            backoff *= 2

async def rerank_all_concurrent_costed(df_llm: pd.DataFrame, concurrency: int = 100):
    """
    Runs reranking concurrently and returns:
      - df_scores: DataFrame aligned to df_llm order
      - totals: dict with summed tokens + cost for THIS run
    """
    sem = asyncio.Semaphore(concurrency)

    async def wrapped(idx, row):
        async with sem:
            return await rerank_one(idx, row)

    tasks = [asyncio.create_task(wrapped(i, row)) for i, row in df_llm.iterrows()]

    totals = {"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}
    results = {}

    t0 = time.monotonic()
    pbar = tqdm(asyncio.as_completed(tasks), total=len(tasks))

    for fut in pbar:
        idx, out, u = await fut
        results[idx] = out

        totals["requests"] += u["requests"]
        totals["input_tokens"] += u["input_tokens"]
        totals["cached_input_tokens"] += u["cached_input_tokens"]
        totals["output_tokens"] += u["output_tokens"]
        totals["cost_usd"] += u["cost_usd"]

        elapsed = time.monotonic() - t0
        pbar.set_postfix({
            "req": totals["requests"],
            "in_tok": totals["input_tokens"],
            "cached": totals["cached_input_tokens"],
            "out_tok": totals["output_tokens"],
            "cost_$": f"{totals['cost_usd']:.4f}",
            "sec": f"{elapsed:.0f}",
        })

    df_scores = pd.DataFrame([results[i] for i in range(len(df_llm))])
    return df_scores, totals

# ---- run it (10 at a time)
df_scores, totals = await rerank_all_concurrent_costed(df_llm, concurrency=100)

print("\n=== Token + cost summary (this run) ===")
print(totals)

df_scores.head()


  0%|          | 0/150 [00:00<?, ?it/s]


=== Token + cost summary (this run) ===
{'requests': 150, 'input_tokens': 136535, 'cached_input_tokens': 0, 'output_tokens': 195575, 'cost_usd': 0.08505674999999999}


,relevance,conceptual_fit,scope_adherence,facet_hits,notes
0,0,0,0,None,Online-focused; poor fit for offline brick-and...
1,25,25,20,None,Abstract unavailable; unsure offline dynamic p...
2,0,0,0,None,Online retail focus; offline brick-and-mortar ...
3,25,20,20,"[1, 5, 7]",Online emphasis; weak offline-focused theoreti...
4,20,20,15,[2],Weak alignment with offline dynamic pricing; a...


In [64]:
mx = df_scores[["relevance","conceptual_fit","scope_adherence"]].max().max()
mn = df_scores[["relevance","conceptual_fit","scope_adherence"]].min().min()
print("Score range:", mn, "to", mx)
df_scores[["relevance","conceptual_fit","scope_adherence","facet_hits","notes"]].head(10)


Score range: 0 to 85


,relevance,conceptual_fit,scope_adherence,facet_hits,notes
0,0,0,0,None,Online-focused; poor fit for offline brick-and...
1,25,25,20,None,Abstract unavailable; unsure offline dynamic p...
2,0,0,0,None,Online retail focus; offline brick-and-mortar ...
3,25,20,20,"[1, 5, 7]",Online emphasis; weak offline-focused theoreti...
4,20,20,15,[2],Weak alignment with offline dynamic pricing; a...
5,85,75,60,"[0, 4, 7]","Strong offline, time-based pricing focus; limi..."
6,75,80,70,[1],Limited abstract; aligns with demand-based pri...
7,25,20,25,"[1, 3, 5]",Weak offline-theory depth; online focus violat...
8,40,35,25,"[2, 4, 5]",Offline focus unclear; abstract unavailable.
9,20,20,20,None,Title alone; missing abstract; offline emphasi...


## C.3

In [65]:
import numpy as np
import pandas as pd

def minmax(x):
    x = np.asarray(x, dtype=float)
    mn, mx = np.nanmin(x), np.nanmax(x)
    return (x - mn) / (mx - mn + 1e-12)

# Join rerank scores to the same rows (df_scores is aligned with df_llm order)
df_llm2 = pd.concat([df_llm.reset_index(drop=True), df_scores.reset_index(drop=True)], axis=1)

# Normalize to 0..1
df_llm2["rel_norm"] = df_llm2["relevance"] / 100.0
df_llm2["concept_norm"] = df_llm2["conceptual_fit"] / 100.0
df_llm2["scope_norm"] = df_llm2["scope_adherence"] / 100.0

# Primary: LLM rubric judgment
W_REL = 0.60
W_CONCEPT = 0.20
W_SCOPE = 0.20
df_llm2["score_stageC3"] = (
    W_REL * df_llm2["rel_norm"] +
    W_CONCEPT * df_llm2["concept_norm"] +
    W_SCOPE * df_llm2["scope_norm"]
)

# Secondary tie-break: your hybrid semantic score (from Option 3 pool rerank)
df_llm2["hybrid_norm"] = minmax(df_llm2["score_hybrid_pool"].values)

df_llm2["final_score"] = 0.90 * df_llm2["score_stageC3"] + 0.10 * df_llm2["hybrid_norm"]

top50 = (
    df_llm2.sort_values("final_score", ascending=False)
    .head(50)
    .reset_index(drop=True)
)

cols = [
    "final_score","relevance","conceptual_fit","scope_adherence",
    "title","year","venue","sources","doi_norm","facet_hits","notes"
]
top50[cols]


,final_score,relevance,conceptual_fit,scope_adherence,title,year,venue,sources,doi_norm,facet_hits,notes
0,0.755323,85,75,60,When should grocery stores adopt time‐based pr...,2023.0,Production and operations management,semantic_scholar,10.1111/poms.14010,"[0, 4, 7]","Strong offline, time-based pricing focus; limi..."
1,0.748066,80,85,85,Technical Note - Intertemporal Price Discrimin...,2016.0,Operational Research,semantic_scholar,10.1287/opre.2015.1473,"[0, 2, 4]",Strong theoretical alignment; offline pricing ...
2,0.728291,75,80,70,Demand-based pricing versus past-price depende...,2006.0,NaN,semantic_scholar,NaN,[1],Limited abstract; aligns with demand-based pri...
3,0.725499,80,70,85,Asymmetric Dynamic Pricing in a Local Gasoline...,2008.0,NaN,semantic_scholar,10.1111/j.1467-6451.2008.00349.x,"[0, 1, 7]",Offline gasoline pricing; uncertain abstract c...
4,0.713239,80,75,72,Retail Pricing and Clearance Sales,1984.0,NaN,openalex,10.3386/w1446,"[0, 1, 2]","Strong offline, theory-focused on time-based p..."
5,0.694272,75,85,70,Dynamic pricing in retail with diffusion proce...,2017.0,IMA Journal of Management Mathematics,semantic_scholar,10.1093/imaman/dpz003,[1],Strong theoretical foundation; offline brick-a...
6,0.674949,72,70,65,Dynamic pricing in the presence of reference p...,2020.0,International Journal of Production Research,semantic_scholar,10.1080/00207543.2019.1598592,"[0, 4, 7]",Theoretical emphasis on reference price effect...
7,0.656501,72,75,65,Optimal Dynamic Pricing for Perishable Assets ...,2000.0,NaN,semantic_scholar,10.1287/mnsc.46.3.375.12063,"[0, 1, 5]",Theoretical pricing framework; potential offli...
8,0.654437,75,70,65,Intertemporal price discrimination: dynamic ar...,2016.0,NaN,semantic_scholar,10.1257/aer.20130564,"[0, 1, 2]",Strong theoretical focus; offline applicabilit...
9,0.644593,75,70,60,Dynamic Pricing for Heterogeneous Time-Sensiti...,2017.0,Manufacturing & Service Operations Management,semantic_scholar,10.2139/ssrn.2992112,"[0, 1, 6]",Strong theoretical focus; offline applicabilit...


In [66]:
# Inspect top 20 titles quickly
top50[["final_score","relevance","scope_adherence","conceptual_fit","title"]].head(20)

,final_score,relevance,scope_adherence,conceptual_fit,title
0,0.755323,85,60,75,When should grocery stores adopt time‐based pr...
1,0.748066,80,85,85,Technical Note - Intertemporal Price Discrimin...
2,0.728291,75,70,80,Demand-based pricing versus past-price depende...
3,0.725499,80,85,70,Asymmetric Dynamic Pricing in a Local Gasoline...
4,0.713239,80,72,75,Retail Pricing and Clearance Sales
5,0.694272,75,70,85,Dynamic pricing in retail with diffusion proce...
6,0.674949,72,65,70,Dynamic pricing in the presence of reference p...
7,0.656501,72,65,75,Optimal Dynamic Pricing for Perishable Assets ...
8,0.654437,75,65,70,Intertemporal price discrimination: dynamic ar...
9,0.644593,75,60,70,Dynamic Pricing for Heterogeneous Time-Sensiti...


In [67]:
# Soft diagnostic: flag top results that mention blueprint negative_query_terms.
neg_terms = [str(t).lower() for t in (blueprint.get("negative_query_terms") or []) if t]
if not neg_terms:
    print("No negative_query_terms in blueprint; skipping this diagnostic.")
else:
    t = (top50["title"].fillna("") + " " + top50["notes"].fillna("")).str.lower()
    flag = t.apply(lambda s: any(w in s for w in neg_terms))
    print("Top50 flagged by negative_query_terms:", int(flag.sum()), "/", len(top50))
    top50.loc[flag, ["final_score","title","notes"]].head(20)


Top50 flagged by negative_query_terms: 0 / 50


In [68]:
top50.to_csv("stageC3_top50_reranked.csv", index=False)
print("Saved: stageC3_top50_reranked.csv")


Saved: stageC3_top50_reranked.csv


# Stage D

In [69]:
import numpy as np
import pandas as pd

# Choose a candidate pool bigger than the final set for diversity selection
MMR_POOL = 200
FINAL_K = 20  # set to 10..20 as you like

cand = df_llm2.sort_values("final_score", ascending=False).head(MMR_POOL).copy().reset_index(drop=True)

# Build text used for similarity (title+abstract). We'll prefer abstract but fall back to title.
cand["mmr_text"] = (cand["title"].fillna("") + "\n\n" + cand["abstract"].fillna("")).astype(str).str.strip()

print("MMR candidate pool:", cand.shape)
cand[["final_score","title"]].head(10)


MMR candidate pool: (150, 46)


,final_score,title
0,0.755323,When should grocery stores adopt time‐based pr...
1,0.748066,Technical Note - Intertemporal Price Discrimin...
2,0.728291,Demand-based pricing versus past-price depende...
3,0.725499,Asymmetric Dynamic Pricing in a Local Gasoline...
4,0.713239,Retail Pricing and Clearance Sales
5,0.694272,Dynamic pricing in retail with diffusion proce...
6,0.674949,Dynamic pricing in the presence of reference p...
7,0.656501,Optimal Dynamic Pricing for Perishable Assets ...
8,0.654437,Intertemporal price discrimination: dynamic ar...
9,0.644593,Dynamic Pricing for Heterogeneous Time-Sensiti...


In [70]:
import os, time
from openai import OpenAI
from pathlib import Path

assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY env var."
client = OpenAI()

EMBED_MODEL = "text-embedding-3-small"
BATCH_SIZE = 64
MAX_CHARS_PER_DOC = 3500

CACHE_DIR = Path(".embed_cache")
CACHE_DIR.mkdir(exist_ok=True)
MMR_EMBED_NPZ = CACHE_DIR / f"mmr_pool_embeds_{EMBED_MODEL}_n{MMR_POOL}.npz"

def truncate_text(s: str, max_chars: int) -> str:
    s = "" if s is None else str(s)
    s = s.replace("\n", " ").strip()
    return s[:max_chars]

def embed_texts(texts, model=EMBED_MODEL, batch_size=BATCH_SIZE, max_retries=6):
    import numpy as np
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        backoff = 1.0
        for attempt in range(1, max_retries + 1):
            try:
                resp = client.embeddings.create(model=model, input=batch)
                vecs = [np.array(item.embedding, dtype=np.float32) for item in resp.data]
                all_vecs.extend(vecs)
                break
            except Exception:
                if attempt == max_retries:
                    raise
                time.sleep(backoff)
                backoff *= 2
    return np.vstack(all_vecs)

def save_npz(path, keys, mat):
    import numpy as np
    np.savez_compressed(path, keys=np.array(keys, dtype=object), embeds=mat.astype(np.float32))

def load_npz(path):
    import numpy as np
    z = np.load(path, allow_pickle=True)
    return list(z["keys"]), z["embeds"].astype(np.float32)

if MMR_EMBED_NPZ.exists():
    keys_cached, embeds = load_npz(MMR_EMBED_NPZ)
    # If cache keys match current pool order, reuse directly; else recompute.
    if keys_cached == cand["merge_key"].astype(str).tolist():
        cand_embeds = embeds
        print("Loaded cached MMR embeddings:", cand_embeds.shape)
    else:
        texts = [truncate_text(t, MAX_CHARS_PER_DOC) for t in cand["mmr_text"].tolist()]
        cand_embeds = embed_texts(texts)
        save_npz(MMR_EMBED_NPZ, cand["merge_key"].astype(str).tolist(), cand_embeds)
        print("Recomputed+saved MMR embeddings:", cand_embeds.shape)
else:
    texts = [truncate_text(t, MAX_CHARS_PER_DOC) for t in cand["mmr_text"].tolist()]
    cand_embeds = embed_texts(texts)
    save_npz(MMR_EMBED_NPZ, cand["merge_key"].astype(str).tolist(), cand_embeds)
    print("Computed+saved MMR embeddings:", cand_embeds.shape)


Recomputed+saved MMR embeddings: (150, 1536)


In [71]:
import os, time
from openai import OpenAI
from pathlib import Path

assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY env var."
client = OpenAI()

EMBED_MODEL = "text-embedding-3-small"
BATCH_SIZE = 64
MAX_CHARS_PER_DOC = 3500

CACHE_DIR = Path(".embed_cache")
CACHE_DIR.mkdir(exist_ok=True)
MMR_EMBED_NPZ = CACHE_DIR / f"mmr_pool_embeds_{EMBED_MODEL}_n{MMR_POOL}.npz"

def truncate_text(s: str, max_chars: int) -> str:
    s = "" if s is None else str(s)
    s = s.replace("\n", " ").strip()
    return s[:max_chars]

def embed_texts(texts, model=EMBED_MODEL, batch_size=BATCH_SIZE, max_retries=6):
    import numpy as np
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        backoff = 1.0
        for attempt in range(1, max_retries + 1):
            try:
                resp = client.embeddings.create(model=model, input=batch)
                vecs = [np.array(item.embedding, dtype=np.float32) for item in resp.data]
                all_vecs.extend(vecs)
                break
            except Exception:
                if attempt == max_retries:
                    raise
                time.sleep(backoff)
                backoff *= 2
    return np.vstack(all_vecs)

def save_npz(path, keys, mat):
    import numpy as np
    np.savez_compressed(path, keys=np.array(keys, dtype=object), embeds=mat.astype(np.float32))

def load_npz(path):
    import numpy as np
    z = np.load(path, allow_pickle=True)
    return list(z["keys"]), z["embeds"].astype(np.float32)

if MMR_EMBED_NPZ.exists():
    keys_cached, embeds = load_npz(MMR_EMBED_NPZ)
    # If cache keys match current pool order, reuse directly; else recompute.
    if keys_cached == cand["merge_key"].astype(str).tolist():
        cand_embeds = embeds
        print("Loaded cached MMR embeddings:", cand_embeds.shape)
    else:
        texts = [truncate_text(t, MAX_CHARS_PER_DOC) for t in cand["mmr_text"].tolist()]
        cand_embeds = embed_texts(texts)
        save_npz(MMR_EMBED_NPZ, cand["merge_key"].astype(str).tolist(), cand_embeds)
        print("Recomputed+saved MMR embeddings:", cand_embeds.shape)
else:
    texts = [truncate_text(t, MAX_CHARS_PER_DOC) for t in cand["mmr_text"].tolist()]
    cand_embeds = embed_texts(texts)
    save_npz(MMR_EMBED_NPZ, cand["merge_key"].astype(str).tolist(), cand_embeds)
    print("Computed+saved MMR embeddings:", cand_embeds.shape)


Loaded cached MMR embeddings: (150, 1536)


In [72]:
import numpy as np
import json
import hashlib
from pathlib import Path

def l2_normalize(mat: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12
    return mat / norms

# Cosine similarity via normalized dot product
E = l2_normalize(cand_embeds)
sim_mat = E @ E.T  # (pool x pool)

# --- facet assignment (for facet-aware diversity)
facet_queries = blueprint.get("facet_queries") or []

cand = cand.copy()
cand["facet_best_i"] = -1
cand["facet_best_query"] = ""

if facet_queries:
    cache_dir = globals().get("CACHE_DIR", Path(".embed_cache"))
    cache_dir.mkdir(exist_ok=True)
    model = globals().get("EMBED_MODEL", "text-embedding-3-small")
    blob = json.dumps({"model": model, "facet_queries": facet_queries}, ensure_ascii=False)
    h = hashlib.sha1(blob.encode("utf-8")).hexdigest()
    facet_embed_path = cache_dir / f"facet_query_embeds_{model}_{h}.json"

    if facet_embed_path.exists():
        q = json.loads(facet_embed_path.read_text(encoding="utf-8"))
        q_facets = np.array(q["embeddings"], dtype=np.float32)
    else:
        if "embed_texts" not in globals():
            raise RuntimeError("Missing embed_texts(). Run the Stage D embedding cell first.")
        batch_size = int(globals().get("BATCH_SIZE", 64))
        q_facets = embed_texts(facet_queries, model=model, batch_size=batch_size)
        facet_embed_path.write_text(
            json.dumps({"model": model, "facet_queries": facet_queries, "embeddings": q_facets.tolist()}, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    Q = l2_normalize(q_facets)
    facet_sim = E @ Q.T  # (MMR_POOL x n_facets)
    facet_best_i = facet_sim.argmax(axis=1).astype(int)
    cand["facet_best_i"] = facet_best_i
    cand["facet_best_query"] = [facet_queries[i] for i in facet_best_i]

relevance = cand["final_score"].to_numpy(dtype=float)
facet_best = cand["facet_best_i"].to_numpy(dtype=int)

def mmr_select(relevance, sim_mat, facet_best=None, k=20, lam=0.75, facet_bonus=0.05):
    """Facet-aware MMR.
    lam close to 1 => prioritize relevance
    lam close to 0 => prioritize diversity
    facet_bonus => small incentive to cover new facet_best labels
    """
    n = len(relevance)
    selected = []
    remaining = set(range(n))
    covered = set()

    # start with highest relevance
    first = int(np.argmax(relevance))
    selected.append(first)
    remaining.remove(first)
    if facet_best is not None and int(facet_best[first]) >= 0:
        covered.add(int(facet_best[first]))

    while len(selected) < k and remaining:
        best_i = None
        best_score = -1e9
        for i in remaining:
            max_sim_to_selected = max(sim_mat[i, j] for j in selected)
            score = lam * relevance[i] - (1 - lam) * max_sim_to_selected
            if facet_best is not None:
                fb = int(facet_best[i])
                if fb >= 0 and fb not in covered:
                    score += facet_bonus
            if score > best_score:
                best_score = score
                best_i = i

        selected.append(best_i)
        remaining.remove(best_i)
        if facet_best is not None:
            fb = int(facet_best[best_i])
            if fb >= 0:
                covered.add(fb)

    return selected

LAMBDA = 0.75
FACET_BONUS = 0.05
picked_idx = mmr_select(relevance, sim_mat, facet_best=facet_best, k=FINAL_K, lam=LAMBDA, facet_bonus=FACET_BONUS)

final_diverse = cand.iloc[picked_idx].copy()
final_diverse = final_diverse.sort_values("final_score", ascending=False).reset_index(drop=True)

final_diverse[["final_score","relevance","scope_adherence","conceptual_fit","facet_best_query","title","year","venue","sources","doi_norm"]]


,final_score,relevance,scope_adherence,conceptual_fit,facet_best_query,title,year,venue,sources,doi_norm
0,0.755323,85,60,75,time-based pricing,When should grocery stores adopt time‐based pr...,2023.0,Production and operations management,semantic_scholar,10.1111/poms.14010
1,0.748066,80,85,85,price discrimination in retail,Technical Note - Intertemporal Price Discrimin...,2016.0,Operational Research,semantic_scholar,10.1287/opre.2015.1473
2,0.728291,75,70,80,demand-based pricing,Demand-based pricing versus past-price depende...,2006.0,NaN,semantic_scholar,NaN
3,0.725499,80,85,70,demand-based pricing,Asymmetric Dynamic Pricing in a Local Gasoline...,2008.0,NaN,semantic_scholar,10.1111/j.1467-6451.2008.00349.x
4,0.713239,80,72,75,price discrimination in retail,Retail Pricing and Clearance Sales,1984.0,NaN,openalex,10.3386/w1446
5,0.694272,75,70,85,demand-based pricing,Dynamic pricing in retail with diffusion proce...,2017.0,IMA Journal of Management Mathematics,semantic_scholar,10.1093/imaman/dpz003
6,0.674949,72,65,70,demand-based pricing,Dynamic pricing in the presence of reference p...,2020.0,International Journal of Production Research,semantic_scholar,10.1080/00207543.2019.1598592
7,0.656501,72,65,75,demand-based pricing,Optimal Dynamic Pricing for Perishable Assets ...,2000.0,NaN,semantic_scholar,10.1287/mnsc.46.3.375.12063
8,0.654437,75,65,70,time-based pricing,Intertemporal price discrimination: dynamic ar...,2016.0,NaN,semantic_scholar,10.1257/aer.20130564
9,0.644593,75,60,70,demand-based pricing,Dynamic Pricing for Heterogeneous Time-Sensiti...,2017.0,Manufacturing & Service Operations Management,semantic_scholar,10.2139/ssrn.2992112


In [73]:
import pandas as pd

facet_queries = blueprint.get("facet_queries") or []

if facet_queries and "facet_best_i" in final_diverse.columns:
    counts = final_diverse["facet_best_i"].value_counts().sort_index()
    df_cov = pd.DataFrame({
        "facet_i": counts.index.astype(int),
        "count": counts.values,
        "facet_query": [facet_queries[i] if 0 <= int(i) < len(facet_queries) else "" for i in counts.index.astype(int)],
    })
    display(df_cov)

final_diverse[["final_score","title","facet_best_query","notes"]].head(30)


,facet_i,count,facet_query
0,0,3,time-based pricing
1,1,12,demand-based pricing
2,2,5,price discrimination in retail


,final_score,title,facet_best_query,notes
0,0.755323,When should grocery stores adopt time‐based pr...,time-based pricing,"Strong offline, time-based pricing focus; limi..."
1,0.748066,Technical Note - Intertemporal Price Discrimin...,price discrimination in retail,Strong theoretical alignment; offline pricing ...
2,0.728291,Demand-based pricing versus past-price depende...,demand-based pricing,Limited abstract; aligns with demand-based pri...
3,0.725499,Asymmetric Dynamic Pricing in a Local Gasoline...,demand-based pricing,Offline gasoline pricing; uncertain abstract c...
4,0.713239,Retail Pricing and Clearance Sales,price discrimination in retail,"Strong offline, theory-focused on time-based p..."
5,0.694272,Dynamic pricing in retail with diffusion proce...,demand-based pricing,Strong theoretical foundation; offline brick-a...
6,0.674949,Dynamic pricing in the presence of reference p...,demand-based pricing,Theoretical emphasis on reference price effect...
7,0.656501,Optimal Dynamic Pricing for Perishable Assets ...,demand-based pricing,Theoretical pricing framework; potential offli...
8,0.654437,Intertemporal price discrimination: dynamic ar...,time-based pricing,Strong theoretical focus; offline applicabilit...
9,0.644593,Dynamic Pricing for Heterogeneous Time-Sensiti...,demand-based pricing,Strong theoretical focus; offline applicabilit...
